# TUTORIAL: The Rijke tube, from model to real-time data assimilation

## 1. The Rijke tube low-order model
### Longitudinal thermoacoustic instabilities

The acoustics in a Rijke tube are governed by the dimensionless linearized
momentum and energy conservation equations, with the time-delayed problem cast
in the Markovian framework of Huhn & Magri (2019). The acoustic velocity and
pressure are expanded on $N_m$ Galerkin modes,

$$
u(x,t)=\sum^{N_m}_{j=1}\eta_j(t)\cos{\left(\frac{\omega_j}{\bar{c}}x\right)},
\qquad
p(x,t)=-\sum^{N_m}_{j=1}\mu_j(t)\sin{\left(\frac{\omega_j}{\bar{c}}x\right)},
$$

and the flame time delay $\tau$ is carried by an advection equation discretized
with $N_c$ Chebyshev collocation points, which acts as a numerical memory of the
acoustic velocity at the heat-source location. The heat release follows a
time-delayed square-root law of intensity $\beta$.

A Rijke tube in the laboratory, with sound: <https://youtu.be/HCguUpaAvX4>.

## The model now lives in `dynamodels`

The model itself, and the tutorial that walks through its state vector, its
acoustic modes and its pressure field, moved to
[`dynamodels`](https://andreanovoa.github.io/dynamodels/), the modelling core
that `romda` builds on:

- **Live tutorial:**
  [The Rijke tube](https://andreanovoa.github.io/dynamodels/tutorials/tutorial_rijke.html)
  — the Galerkin and Chebyshev discretizations, the acoustic modes, the delayed
  flame velocity, the microphone pressures, and an animation of the pressure
  field along the tube.
- **Model reference:**
  [Rijke tube](https://andreanovoa.github.io/dynamodels/models/rijke/) — the
  named regimes (`case=`) along the $\beta$ route and the full API.

Section 2 below assimilates data into this model, which is what `romda` adds on
top, and [21](21_TABADA_Rijke_CMAME.ipynb) goes on to correct the model bias.

`romda.models.physical` re-exports the `dynamodels` models, so existing code
keeps working unchanged:

In [ ]:
from romda.models.physical import Rijke

case = Rijke()  # beta = 4: a period-2 limit cycle

state, t_ = case.time_integrate(int(case.t_transient / case.dt))
case.update_history(state, t_)
case.visualize_observable_hist()

## 2. Real-time data assimilation

### Augmented state

We perform state and parameter estimation in the Rijke tube model of section 1. 
The augmented state space vector of the Rijke tube model is
$$
\boldsymbol{\psi} = \begin{bmatrix}
        \boldsymbol{\phi}\\
        \boldsymbol{\alpha}\\
        \mathbf{p}_{mic}
        \end{bmatrix}
        =
        \begin{bmatrix}
        \boldsymbol{\eta} \\ 
        \boldsymbol{\mu} \\ 
        \boldsymbol{\nu} \\ 
        \beta\\
        \tau\\
        \mathbf{p}_{mic}
        \end{bmatrix}
        \in \mathbb{R}^{2N_m+N_c+2+N_q},
$$
where 
- $\boldsymbol{\eta}\in \mathbb{R}^{N_m}$: acoustic velocity modes (from Galerkin projection)
- $\boldsymbol{\mu}\in \mathbb{R}^{N_m}$: acoustic pressure modes (from Galerkin projection)
- $\boldsymbol{\nu}\in \mathbb{R}^{N_c}$: advection "velocity" modes (from Chevyshev projection)
- $\beta$: heat source strength
- $\tau$: acoustic time delay
- $\mathbf{p}_{mic} \mathbb{R}^{N_q}$: acoustic pressure at the microphone locations computed as
$$
{p}_{mic}(x_q, t) = -\sum^{N_m}_{j=1}\,\mu_j(t)\sin{\left(\dfrac{\omega_j}{\bar{c}} x_q\right)}. 
$$


In [ ]:
from romda.observations import Observations
from romda.models.physical import Rijke


truth = Observations(model=Rijke,
                     t_start=.2,
                     t_stop=.6,
                     Nt_obs=20,
                     beta=3.2,
                     tau=1.2e-3,
                     add_noise=True,
                     noise_type='pink, add',
                     noise_level=0.25,
                     )

truth.plot_truth(truth, f_max=2000, window=0.02, fig_width=12)

In [ ]:
from romda.estimators import EnKF, EnSRKF


ensemble = EnKF(parent_model=Rijke(dt=truth.dt),
                    m=10,
                    std_phi=0.25,
                    std_alpha=dict(beta=[3., 4.],
                                   tau=[1e-3, 2e-3]),
                    distribution_alpha='uniform',
                    inflation_factor=1.0,
                    inflation_factor_rejection=1.005,
                    )

ensemble.visualize_state()

In [ ]:
import numpy as np

filter_ens = ensemble.copy()

# Observation error covariance matrix
std_obs = 0.1
Cdd = np.diag(std_obs * np.ones(filter_ens.model.Nq)) * np.max(abs(truth.y_obs), axis=0) ** 2

# Assimilate all the observations sequentially: forecast to the observation time and analyse
for d, t_d in zip(truth.y_obs, truth.t_obs):
    filter_ens.forecast_step(t_end=t_d)
    filter_ens.analysis_step(d=d, Cdd=Cdd.copy())

# Forecast a bit further after the last observation and close the multiprocessing pools
filter_ens.forecast_step(t_end=truth.t_obs[-1] + 10 * filter_ens.model.t_CR, close=True)

In [ ]:
filter_ens.visualize_history(truth=truth, plot_members=True, dims=[0, 1])

In [ ]:
filter_ens.visualize_history(truth=truth, plot_members=False, reference_a=truth.true_parameters)